CNN Architecture and Implementation

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [4]:
#Datasets & DataLoaders 

from torch.utils.data import DataLoader
import torchvision.transforms as transforms 

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5) , (0.5,0.5,0.5))
])

trainset = CIFAR10(root="./data" , train=True, download=True , transform=transform)
testset = CIFAR10(root="./data" , train=False,download=True , transform=transform)

In [5]:
trainloader = DataLoader(trainset , batch_size = 64 , shuffle = True)
testloader = DataLoader(testset , batch_size=64)

CNN Build


In [6]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN , self).__init__()
        
        self.convo_layers = nn.Sequential(
            nn.Conv2d(3 , 32 ,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2) ,
            
            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2) , 
            
            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2) 
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128 , 256),
            nn.ReLU(),
            nn.Linear(256 , 10)
        )
    
    def forward(self , x):
        x = self.convo_layers(x)
        x = x.view(x.size(0) ,-1) # flattening
        x = self.fc_layers(x)
        
        return x
        


In [7]:
model = CNN()
print(model)

CNN(
  (convo_layers): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc_layers): Sequential(
    (0): Linear(in_features=2048, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=10, bias=True)
  )
)


In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

Training the CNN Models

In [10]:
epochs = 10 

for epoch in range(epochs) :
    epoch_training_loss = 0.0
    
    for images, labels in trainloader :
        optimizer.zero_grad()
        
        output = model.forward(images) # FP
        loss = criterion(output,labels)
        loss.backward() #BP
        optimizer.step() #update parameters
        
        epoch_training_loss += loss.item()
    
    print(f"epoch = {epoch + 1}/ {epochs} & loss = {epoch_training_loss/len(trainloader)}")

epoch = 1/ 10 & loss = 1.1793575142807973


KeyboardInterrupt: 

In [11]:
# Evalution of CNN

correct_labels = 0 
total_labels = 0 

model.eval()

with torch.no_grad():
    for images , labels in testloader :
        outputs = model.forward(images)
        _, predicted = torch.max(outputs,1)
        
        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)


print(f"Accuracy = {correct_labels / total_labels * 100} %")

Accuracy = 65.53 %
